# Huffman Coding (Greedy Approach)

## What is Huffman Coding?
Huffman coding is a **lossless data compression** algorithm. It assigns a **variable-length binary code** to each character:
- **Frequent** characters get **short** codes.
- **Rare** characters get **long** codes.

This reduces the average number of bits per character compared with fixed-length encodings such as ASCII (8 bits per character).

## Key Property: Prefix-Free Codes
No code is a prefix of another code, so an encoded bit stream can be decoded **unambiguously** without separators.

## Greedy Strategy
1. Count the frequency of every character.
2. Create a leaf node for each character and put all nodes into a **min-heap** keyed by frequency.
3. Repeatedly remove the **two nodes with the smallest frequencies** and merge them into a new parent node whose frequency is their sum.
4. Push the parent back into the heap, and repeat until a single node (the **root**) remains.
5. Traverse the tree: going **left appends `0`** and going **right appends `1`**. The path to each leaf is that character's code.

## Step 1: Import Libraries

| Library | Purpose |
|---|---|
| `heapq` (as `hq`) | Min-heap (priority queue) to always retrieve the lowest-frequency node efficiently |
| `collections.Counter` (as `cntr`) | Counts character frequencies in the input string |
| `numpy` (as `np`) | Imported but **not used** in this notebook, so it can be safely removed |

In [ ]:
import numpy as np
import heapq as hq
from collections import Counter as cntr


## Step 2: Define the `Node` Class

Each node in the Huffman tree stores:

| Attribute | Meaning |
|---|---|
| `char` | The character (for **leaf** nodes) or `None` (for **internal** nodes) |
| `freq` | The frequency of the character, or the combined frequency of its subtree |
| `left`, `right` | Child nodes (`None` for leaves) |

### Why `__lt__`?
`heapq` compares objects using the `<` operator. By defining `__lt__` to compare `freq`, the heap will always keep the node with the **smallest frequency** at the top.

> **Note:** when two nodes have equal frequencies, their relative order depends on heap internals. The resulting codes may differ between implementations, but the **total encoded length is always the same (optimal)**.

In [ ]:
class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None
    def __lt__(self, other):
        return self.freq < other.freq

## Step 3: Build the Huffman Tree — `tree(s)`

**Input:** a string `s`
**Output:** the root `Node` of the Huffman tree (or `None` for an empty string)

### Steps
1. `cntr(s)` produces a frequency table, e.g. `{'i': 7, 'e': 5, ...}`.
2. A leaf `Node` is created for every `(character, frequency)` pair.
3. `hq.heapify(heap)` converts the list into a min-heap in O(k).
4. While more than one node remains:
   - Pop the **two smallest** nodes (`l` and `r`).
   - Create a new internal node (`char = None`) with `freq = l.freq + r.freq`.
   - Attach `l` as the left child and `r` as the right child.
   - Push the new node back into the heap.
5. The single remaining node, `heap[0]`, is the **root** of the tree.

### Notes
- The variable `ans = {}` is declared but never used, so it can be deleted.
- A string with only **one unique character** gives a single-node tree, so that character's code will be an empty string `""`. Handle this case separately if you need a code of `"0"`.

In [ ]:
def tree(s):
    fr = cntr(s)
    heap = [Node(ch, f) for ch, f in fr.items()]
    hq.heapify(heap)
    ans = {}
    while(len(heap) >  1):
        l = hq.heappop(heap)
        r = hq.heappop(heap)

        nxt = Node(None,l.freq + r.freq)
        nxt.left = l
        nxt.right = r
        hq.heappush(heap,nxt)
    return heap[0] if heap else None


## Step 4: Generate the Codes — `HuffCodes(node, curr="", temp=None)`

This function walks the tree **recursively** (a depth-first, pre-order traversal) and records the path to each leaf.

| Parameter | Meaning |
|---|---|
| `node` | Current node being visited |
| `curr` | The bit string accumulated along the path so far |
| `temp` | Dictionary that maps `character → code` |

### Logic
- **Base case:** if `node is None`, return without doing anything.
- **Leaf node** (`node.char is not None`): store `temp[node.char] = curr`.
- **Recursive case:** go **left** with `curr + "0"` and **right** with `curr + "1"`.
- `temp=None` is used instead of `temp={}` as the default argument to avoid Python's *mutable default argument* pitfall, and a fresh dictionary is created on the first call.

The finished dictionary is returned to the caller.

In [ ]:
def HuffCodes(node,curr="",temp = None):
    if temp is None:
        temp = {}
        
    if node is not None:
        if node.char is not None:
            temp[node.char] = curr
        HuffCodes(node.left, curr+"0",temp)
        HuffCodes(node.right, curr+"1",temp)

    return temp

## Step 5: Run the Algorithm

We build a Huffman tree for the following input string and print the resulting code table:

```
ieqyfuqioeiuqegtuoyweqoiuriuq3ewwgifgru3iyfgruy3gfru2qky3tr
```

Characters that appear many times (such as `u`, `q`, `i`) should receive **shorter** codes, and rare ones (such as `2`, `k`, `t`) should receive **longer** codes.

In [ ]:
root = tree("ieqyfuqioeiuqegtuoyweqoiuriuq3ewwgifgru3iyfgruy3gfru2qky3tr")
codes = HuffCodes(root)
print(codes)

## Results

The output is a dictionary mapping each unique character to its Huffman code, for example:

```
{'q': '010', 'u': '110', ..., '2': '00110', ...}
```
*(The exact codes in your output may differ from this illustration, because ties in frequency can be broken in different ways. Use your own printed output as the source of truth.)*

### How to read the output
- Each **key** is a character in the input string.
- Each **value** is its binary code (`0` = left branch, `1` = right branch).
- More frequent characters have **shorter** codes.
- The codes are **prefix-free**, so no code is the beginning of another.

### Compression benefit
For a string of length *n* with a fixed 8-bit encoding, the size is `8n` bits. With Huffman coding, the size is:

`Σ (frequency of char × length of its code)`

which is always **minimal** among all prefix-free codes.

## Complexity Analysis

Let **n** be the length of the string and **k** the number of **unique** characters.

| Step | Time |
|---|---|
| Counting frequencies (`Counter`) | O(n) |
| Creating nodes + `heapify` | O(k) |
| `k − 1` merges, each with 2 pops and 1 push | **O(k log k)** |
| Generating codes (tree traversal) | O(k) node visits (string concatenation adds up to O(k·h) in the worst case, where h is the tree height) |
| **Total** | **O(n + k log k)** |

**Space complexity: O(k)**, made up of:
- The frequency table: O(k)
- The heap and tree, which has `2k − 1` nodes: O(k)
- The code dictionary: O(k) entries, each up to O(h) long
- The recursion stack: O(h), which is O(log k) for a balanced tree and O(k) in the worst case

> For ASCII text, k ≤ 128 (or 256), so k is effectively a constant and the running time is dominated by the O(n) counting step.